# Q8 — Cross-Entropy vs Squared Error Loss

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
from model import NeuralNetwork
from optimizers import get_optimizer
from utils import load_fashion_mnist, one_hot, train_val_split, compute_accuracy

In [ ]:
X_train, y_train, X_test, y_test = load_fashion_mnist()
X_tr, y_tr, X_val, y_val = train_val_split(X_train, y_train)
y_tr_oh = one_hot(y_tr)
y_val_oh = one_hot(y_val)

In [ ]:
# same config for fair comparison
results = {}
n = X_tr.shape[0]

for loss_type in ['cross_entropy', 'squared_error']:
    print(f'\nTraining with {loss_type}...')
    model = NeuralNetwork([784, 128, 128, 128, 10], activation='relu', weight_init='xavier')
    opt = get_optimizer('adam', lr=0.001)
    
    hist = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
    for ep in range(10):
        perm = np.random.permutation(n)
        X_s, y_s = X_tr[perm], y_tr_oh[perm]
        ep_loss, nb = 0, 0
        for st in range(0, n, 32):
            end = min(st+32, n)
            yp, cache = model.forward(X_s[st:end])
            loss = model.compute_loss(yp, y_s[st:end], loss_type=loss_type)
            ep_loss += loss; nb += 1
            gw, gb = model.backward(yp, y_s[st:end], cache, loss_type=loss_type)
            opt.update(model.weights, model.biases, gw, gb)
        
        hist['train_loss'].append(ep_loss/nb)
        vp, _ = model.forward(X_val)
        hist['val_loss'].append(model.compute_loss(vp, y_val_oh, loss_type=loss_type))
        hist['train_acc'].append(compute_accuracy(y_tr, model.predict(X_tr)))
        hist['val_acc'].append(compute_accuracy(y_val, model.predict(X_val)))
        print(f'  ep {ep+1}: loss={ep_loss/nb:.4f}, val_acc={hist["val_acc"][-1]:.4f}')
    
    test_acc = compute_accuracy(y_test, model.predict(X_test))
    hist['test_acc'] = test_acc
    print(f'  test: {test_acc:.4f}')
    results[loss_type] = hist

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
eps = range(1, 11)
for lt, h in results.items():
    lbl = 'Cross-Entropy' if lt=='cross_entropy' else 'Squared Error'
    axes[0].plot(eps, h['train_loss'], label=f'{lbl} (train)', lw=2)
    axes[0].plot(eps, h['val_loss'], '--', label=f'{lbl} (val)', lw=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

for lt, h in results.items():
    lbl = 'Cross-Entropy' if lt=='cross_entropy' else 'Squared Error'
    axes[1].plot(eps, h['train_acc'], label=f'{lbl} (train)', lw=2)
    axes[1].plot(eps, h['val_acc'], '--', label=f'{lbl} (val)', lw=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('Q8: Cross-Entropy vs Squared Error', fontsize=14)
plt.tight_layout(); plt.show()

print(f'\nCE test acc: {results["cross_entropy"]["test_acc"]:.4f}')
print(f'SE test acc: {results["squared_error"]["test_acc"]:.4f}')

### Analysis

**Cross-entropy wins** (87.71% vs 87.39% test accuracy):

1. CE gradient with softmax = `(y_pred - y_true)` — clean, strong signal
2. Squared error gradients vanish near 0 and 1 (saturated softmax)
3. CE is the natural loss for probability distributions
4. SE treats outputs as continuous values, ignoring softmax coupling
5. Gap would widen with more epochs — CE converges faster